# Fine-tune MiniLM for IT Ticket Routing

Fine-tunes `all-MiniLM-L6-v2` as a 7-class sequence classifier on the Kaggle IT ticket dataset.  
**Runtime → Change runtime type → T4 GPU** before running.

Expected training time: ~25 min · Expected macro-F1: 0.88–0.91

In [ ]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
!pip install -q transformers datasets scikit-learn huggingface_hub \
               pandas numpy joblib accelerate evaluate

In [ ]:
# ── Cell 2: Mount Drive ───────────────────────────────────────────────────────
# Put tickets.csv at the root of your Google Drive (MyDrive/tickets.csv)
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
os.makedirs('/content/data', exist_ok=True)
shutil.copy('/content/drive/MyDrive/tickets.csv', '/content/data/tickets.csv')
print('✓ tickets.csv copied')

In [ ]:
# ── Cell 3: Preprocess ────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

CATEGORY_MAP = {
    'Hardware':               'Infrastructure',
    'Administrative rights':  'Access Management',
    'Access':                 'Access Management',
    'Storage':                'Storage',
    'HR Support':             'HR Support',
    'Purchase':               'Procurement',
    'Internal Project':       'Internal Project',
    'Miscellaneous':          'General IT',
}
CATEGORIES = sorted(set(CATEGORY_MAP.values()))
LABEL2ID   = {c: i for i, c in enumerate(CATEGORIES)}
ID2LABEL   = {i: c for c, i in LABEL2ID.items()}

df = pd.read_csv('/content/data/tickets.csv')
df.columns = [c.strip() for c in df.columns]
text_col  = next(c for c in df.columns if 'document' in c.lower() or 'text' in c.lower())
label_col = next(c for c in df.columns if 'topic' in c.lower() or 'label' in c.lower())
df = df[[text_col, label_col]].rename(columns={text_col: 'text', label_col: 'raw_label'})
df['text']     = df['text'].astype(str).str.strip()
df['label']    = df['raw_label'].map(CATEGORY_MAP)
df = df.dropna(subset=['label', 'text'])
df = df[df['text'].str.len() > 5]
df['label_id'] = df['label'].map(LABEL2ID)

print(f'{len(df):,} tickets · {df["label"].nunique()} classes')
print(df['label'].value_counts().to_string())

In [ ]:
# ── Cell 4: Train / val / test split ─────────────────────────────────────────
train_df, tmp     = train_test_split(df, test_size=0.20, stratify=df['label_id'], random_state=42)
val_df,   test_df = train_test_split(tmp, test_size=0.50, stratify=tmp['label_id'], random_state=42)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'train {len(train_df):,}  val {len(val_df):,}  test {len(test_df):,}')

In [ ]:
# ── Cell 5: Tokenise ──────────────────────────────────────────────────────────
from transformers import AutoTokenizer
from datasets import Dataset

BASE_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer  = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenise(df):
    ds = Dataset.from_dict({'text': df['text'].tolist(), 'labels': df['label_id'].tolist()})
    return ds.map(
        lambda b: tokenizer(b['text'], truncation=True, padding='max_length', max_length=128),
        batched=True, batch_size=512
    )

train_ds = tokenise(train_df)
val_ds   = tokenise(val_df)
test_ds  = tokenise(test_df)
print('✓ Tokenised')

In [ ]:
# ── Cell 6: Fine-tune ─────────────────────────────────────────────────────────
import evaluate as hf_evaluate
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

metric = hf_evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels, average='macro')

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(CATEGORIES),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)

args = TrainingArguments(
    output_dir='/content/finetuned_minilm',
    num_train_epochs=5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=100,
    fp16=True,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()
print(f'\nBest val macro-F1: {trainer.state.best_metric:.4f}')

In [ ]:
# ── Cell 7: Evaluate on test set ──────────────────────────────────────────────
from sklearn.metrics import classification_report

preds_out = trainer.predict(test_ds)
preds     = np.argmax(preds_out.predictions, axis=-1)
labels    = preds_out.label_ids

print(classification_report(labels, preds, target_names=CATEGORIES))

In [ ]:
# ── Cell 8: Push to HuggingFace ───────────────────────────────────────────────
# Log in first — paste your HF token when prompted
!huggingface-cli login

HF_REPO = 'starlord0104/ticket-routing-minilm-finetuned'

trainer.model.push_to_hub(HF_REPO, private=False)
tokenizer.push_to_hub(HF_REPO, private=False)

print(f'\n✓ Model live at https://huggingface.co/{HF_REPO}')

In [ ]:
# ── Cell 9: Sanity check ──────────────────────────────────────────────────────
from transformers import pipeline

clf = pipeline('text-classification', model=HF_REPO, top_k=1)

tests = [
    ('Procurement',    'I need to purchase a new laptop, mine is 5 years old.'),
    ('Infrastructure', 'My laptop screen is black and will not turn on.'),
    ('HR Support',     'New hire starting Monday — set up email and system access.'),
    ('Access Mgmt',   'I am locked out of my admin account, need a password reset.'),
    ('Storage',        'Drive is full, I cannot save any files.'),
    ('General IT',     'The office wifi has been very slow all morning.'),
    ('Int. Project',  'Please reassign the Q3 project tasks to the new team lead.'),
]

print(f'{"Expected":20s}  {"Predicted":20s}  {"Conf":6s}  Text')
print('-' * 80)
for expected, text in tests:
    result = clf(text)[0][0]
    match  = '✓' if result['label'] == expected.replace('Mgmt', 'Management').replace('Int. ', 'Internal ') else '✗'
    print(f'{expected:20s}  {result["label"]:20s}  {result["score"]:.2%}  {match}  {text[:45]}')